# DMRG 02: Ground-State Workflow

This notebook runs a complete ground-state DMRG calculation. The workflow is: build an MPO, initialize an MPS, define a sweep schedule, run `DMRG`, and inspect the energy trace and final MPS.


## Workflow map

`MPO builder -> MPS initial state -> Sweeps schedule -> DMRG solver -> energies and optimized MPS`

The examples use a small transverse-field Ising chain so the notebook runs quickly on CPU.


In [1]:
from pathlib import Path
import importlib
import shutil
import subprocess
import sys
import types


def _ensure_ipython_display_if_missing():
    try:
        import IPython.display  # noqa: F401
        return
    except ModuleNotFoundError:
        pass

    ipython_module = types.ModuleType("IPython")
    display_module = types.ModuleType("IPython.display")
    display_module.display = lambda *args, **kwargs: None
    display_module.Math = lambda *args, **kwargs: None
    ipython_module.display = display_module
    ipython_module.get_ipython = lambda: None
    ipython_module.version_info = (0, 0)
    sys.modules.setdefault("IPython", ipython_module)
    sys.modules.setdefault("IPython.display", display_module)


def _import_quantum_simulation():
    _ensure_ipython_display_if_missing()
    return importlib.import_module("quantum_simulation")


try:
    qs = _import_quantum_simulation()
except ModuleNotFoundError as exc:
    if exc.name != "quantum_simulation":
        raise
    repo_dir = Path("quantum-simulation")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", "https://github.com/ToelUl/quantum-simulation.git"], check=True)
    target = Path("quantum_simulation")
    if not target.exists():
        shutil.copytree(repo_dir / "quantum_simulation", target)
    qs = _import_quantum_simulation()

print("Loaded quantum_simulation from", Path(qs.__file__).parent)


Loaded quantum_simulation from /mnt/d/phd_research_projects/quantum-simulation-main/quantum_simulation


In [2]:
import torch
from quantum_simulation import MPS, DMRG, Sweeps, Ising1DMPOBuilder

device = "cpu"
dtype = torch.complex128
print(f"Using device={device}, dtype={dtype}")


Using device=cpu, dtype=torch.complex128


## Step 1: Build the Hamiltonian as an MPO

The 1D transverse-field Ising builder represents the Hamiltonian `H = -J sum Sx_i Sx_{i+1} - g sum Sz_i`. Its `get_mpo()` method returns the list of rank-4 tensors expected by the DMRG driver.


In [3]:
num_sites = 6
j_coupling = 1.0
g_field = 0.7

ising_builder = Ising1DMPOBuilder(
    num_sites=num_sites,
    j_coupling=j_coupling,
    g_field=g_field,
    device=device,
    dtype=dtype,
)
mpo = ising_builder.get_mpo()

print(f"Number of MPO tensors: {len(mpo)}")
print("MPO shapes:")
for site, tensor in enumerate(mpo):
    print(f"  W[{site}] = {tuple(tensor.shape)}")


Number of MPO tensors: 6
MPO shapes:
  W[0] = (1, 3, 2, 2)
  W[1] = (3, 3, 2, 2)
  W[2] = (3, 3, 2, 2)
  W[3] = (3, 3, 2, 2)
  W[4] = (3, 3, 2, 2)
  W[5] = (3, 1, 2, 2)


## Step 2: Initialize the variational MPS

The initial bond dimension does not need to be the final bond dimension. DMRG can grow or truncate bonds according to the `Sweeps` schedule.


In [4]:
initial_mps = MPS(
    Nsites=num_sites,
    chid=2,
    initial_bond_dim=4,
    device=device,
    dtype=dtype,
    seed=2024,
)
print(initial_mps)


Initializing MPS as a right-canonical random state...
Initialization complete. Ortho center at site 0.
MPS(Nsites=6, chid=2, ortho_center=0, bond_dims=[2, 4, 4, 4, 2, 1])


## Step 3: Define a sweep schedule

`Sweeps` controls the optimization budget. Important fields are:

- `maxdim`: maximum MPS bond dimension for each sweep.
- `noise`: random perturbation added after local optimization, often useful in early sweeps.
- `krylov_dim`: size of the Lanczos Krylov space used inside each two-site update.
- `cutoff`: SVD truncation tolerance.
- `reortho` and `maxreortho`: optional Lanczos reorthogonalization settings.


In [5]:
sweeps = Sweeps(3)
sweeps.set_schedule(
    maxdim=[4, 8, 12],
    noise=[1e-5, 1e-6, 0.0],
    krylov_dim=[4, 6, 8],
    cutoff=[1e-8, 1e-10, 1e-12],
    reortho=[False, False, True],
    maxreortho=[1, 1, 1],
)
print(sweeps)


Sweeps(3):
  maxdim: [4, 8, 12]
  noise: [1e-05, 1e-06, 0.0]
  krylov_dim: [4, 6, 8]
  cutoff: [1e-08, 1e-10, 1e-12]
  reortho: [False, False, True]
  maxreortho: [1, 1, 1]



## Step 4: Run DMRG

The DMRG module owns the MPO, applies the effective two-site Hamiltonian, calls the Lanczos eigensolver, and updates the MPS tensors after each SVD truncation.


In [6]:
solver = DMRG(
    initial_mps,
    mpo,
    device=device,
    dtype=dtype,
    contract_backend="auto",
)

energies, optimized_mps = solver(sweeps, dispon=1, maxit=2)

print(f"Number of local updates: {len(energies)}")
print(f"Final energy: {energies[-1]:.12f}")
print(f"Final energy per site: {energies[-1] / num_sites:.12f}")
print(optimized_mps)


[DMRG] Contraction backend resolved to 'ncon' on device='cpu'.
Sweep: 1 of 3, Energy: -2.327594409224, Bond dim: 4, Noise: 1.00e-05, Cutoff: 1.00e-08
Sweep: 2 of 3, Energy: -2.327594419160, Bond dim: 4, Noise: 1.00e-06, Cutoff: 1.00e-10
Sweep: 3 of 3, Energy: -2.327594419162, Bond dim: 4, Noise: 0.00e+00, Cutoff: 1.00e-12
Number of local updates: 30
Final energy: -2.327594419162
Final energy per site: -0.387932403194
MPS(Nsites=6, chid=2, ortho_center=0, bond_dims=[2, 4, 6, 4, 2, 1])


## Step 5: Inspect the energy trace

`energies` stores one value per local two-site optimization. It is a useful diagnostic for convergence and for comparing sweep schedules.


In [7]:
for step, energy in enumerate(energies, start=1):
    print(f"update {step:02d}: E = {energy:.12f}")

best_energy = min(energies)
last_energy = energies[-1]
print(f"Best energy in trace: {best_energy:.12f}")
print(f"Last minus best: {last_energy - best_energy:.3e}")


update 01: E = -2.016674738093
update 02: E = -2.126051869283
update 03: E = -2.325672145611
update 04: E = -2.327379533257
update 05: E = -2.327397543259
update 06: E = -2.327397569236
update 07: E = -2.327399963239
update 08: E = -2.327593857804
update 09: E = -2.327594406488
update 10: E = -2.327594409224
update 11: E = -2.327594409226
update 12: E = -2.327594409598
update 13: E = -2.327594419162
update 14: E = -2.327594419162
update 15: E = -2.327594419160
update 16: E = -2.327594419160
update 17: E = -2.327594419162
update 18: E = -2.327594419162
update 19: E = -2.327594419161
update 20: E = -2.327594419160
update 21: E = -2.327594419160
update 22: E = -2.327594419161
update 23: E = -2.327594419162
update 24: E = -2.327594419162
update 25: E = -2.327594419162
update 26: E = -2.327594419162
update 27: E = -2.327594419162
update 28: E = -2.327594419162
update 29: E = -2.327594419162
update 30: E = -2.327594419162
Best energy in trace: -2.327594419162
Last minus best: 3.948e-13


## Inspect the optimized MPS

The optimized MPS stores both site tensors and Schmidt weights. The shapes below show how the bond dimensions changed during the sweep schedule.


In [8]:
print("A tensor shapes:")
for site, tensor in enumerate(optimized_mps.A):
    print(f"  A[{site}] = {tuple(tensor.shape)}")

print("Schmidt weight shapes and norms:")
for bond, weights in enumerate(optimized_mps.sWeight):
    norm = torch.linalg.norm(weights).item()
    print(f"  sWeight[{bond}] shape={tuple(weights.shape)}, norm={norm:.6f}")


A tensor shapes:
  A[0] = (1, 2, 2)
  A[1] = (2, 2, 4)
  A[2] = (4, 2, 6)
  A[3] = (6, 2, 4)
  A[4] = (4, 2, 2)
  A[5] = (2, 2, 1)
Schmidt weight shapes and norms:
  sWeight[0] shape=(1, 1), norm=1.000000
  sWeight[1] shape=(2,), norm=1.000000
  sWeight[2] shape=(4,), norm=1.000000
  sWeight[3] shape=(6,), norm=1.000000
  sWeight[4] shape=(4,), norm=1.000000
  sWeight[5] shape=(2,), norm=1.000000
  sWeight[6] shape=(1, 1), norm=1.000000


## Takeaways

The full DMRG workflow is compact once the objects are connected: the MPO defines the Hamiltonian, the MPS defines the variational state, `Sweeps` defines the optimization schedule, and `DMRG` returns the local energy history plus the optimized state.
